# 07 Apply Views — Integration + Datamart

**Doel:** Toepassen van alle standaard SQL-views in `views/integration/*.sql` en
`views/datamart/*.sql`.  Views kunnen niet binnen DLT bestaan (DLT beheert
Materialised Views en Streaming Tables, niet plain views), dus deze notebook is
de tegenhanger van de DLT-pipelines voor de gevirtualiseerde laag.

## Wat doet deze notebook?

1. Zet de actieve catalog via `USE CATALOG`.
2. Itereert over `views/integration/*.sql` en voert elk bestand uit via `spark.sql()`.
3. Idem voor `views/datamart/*.sql`.

## Volgorde in de workflow

```
setup
 ├─→ ingest_full
 └─→ ingest_incremental
       ↓
       dlt_integration                  (Streaming Tables in INTEGRATION)
        ↓
        apply_views                     (deze notebook — Integration + Datamart dim views)
         ↓
         dlt_datamart                   (fact_order MV + fact_sales_line MV — leest sales_line view)
```

## Patroon-overzicht in de stack

| Object | Patroon | Reden |
|---|---|---|
| `STAGING_AZURESTORAGE.*` | Delta Table | Landing zone, CDF, Time Travel |
| `INTEGRATION.order_*` + quarantines | Streaming Table | apply_changes state, MERGE |
| `INTEGRATION.sales_line` | **View** | Pure join, geen state, altijd vers |
| `DATAMART.dim_*` (9 stuks) | **View** | Lage cardinaliteit, simpele projecties |
| `DATAMART.fact_order` | MV (in `dlt_datamart`) | Order-grain fact, materialised zodat Silver-correcties automatisch propageren |
| `DATAMART.fact_sales_line` | MV (Liquid Clustering, in `dlt_datamart`) | Zwaarste tabel, profiteert van clustering |

In [ ]:
# Parameters & widgets
dbutils.widgets.text("catalog", "DEMO", "Catalog")
catalog = dbutils.widgets.get("catalog")
print(f"Catalog: {catalog}")

In [ ]:
import os

# DBR 14+ Workspace Files: relative paths resolve from the notebook's folder.
# views/
# ├── 07_apply_views.ipynb   (this notebook — cwd here)
# ├── integration/*.sql
# └── datamart/*.sql

spark.sql(f"USE CATALOG {catalog}")
print(f"Active catalog set to: {catalog}")

## Integration views

Wordt eerst toegepast omdat de Datamart-views (en de `dlt_datamart` MV) van
`INTEGRATION.sales_line` afhankelijk zijn.

In [ ]:
def apply_sql_folder(folder: str) -> None:
    """Voer alle .sql-bestanden in alfabetische volgorde uit binnen `folder`."""
    sql_files = sorted(f for f in os.listdir(folder) if f.endswith(".sql"))
    if not sql_files:
        print(f"  (geen .sql-bestanden in {folder})")
        return
    for name in sql_files:
        path = os.path.join(folder, name)
        with open(path, "r", encoding="utf-8") as f:
            sql = f.read()
        spark.sql(sql)
        print(f"  applied: {path}")

print("Integration views:")
apply_sql_folder("integration")

## Datamart views

In [ ]:
print("Datamart views:")
apply_sql_folder("datamart")

## Validatie

Lijst alle views in INTEGRATION + DATAMART om te bevestigen dat alles is
aangemaakt.

In [ ]:
for schema in ("INTEGRATION", "DATAMART"):
    rows = spark.sql(f"SHOW VIEWS IN {catalog}.{schema}").collect()
    print(f"\n{catalog}.{schema} — {len(rows)} view(s):")
    for r in rows:
        print(f"  {r['viewName']}")